# Notebook 本地测试：批量 PDF -> PaddleOCR -> 分块 -> Milvus -> Dify 本地文件

这一版只保留 notebook 调用，不提供 CLI 或业务接口封装。目标是先在本地验证：

1. 读取一个或一批 PDF。
2. 对扫描版 PDF 使用本地开源 PaddleOCR（依赖 `paddlepaddle==3.3.0`）做 OCR。
3. 按文档类型分块。
4. 可选写入 Milvus（本地 BGE embedding）。
5. 导出本地 Dify 文件，之后手工上传 Dify 测试。

> 默认先跑一个 PDF；批量模式只需把 `PDF_FILES` 放多个文件。


## 0. 安装依赖

OCR 使用开源、可本地部署的 PaddleOCR。请在 notebook 环境中安装：

```bash
pip install paddlepaddle==3.3.0 paddleocr pymupdf
```

`pymupdf` 用于把 PDF 页面渲染成图片，再交给 PaddleOCR。


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

PDF_FILES = [Path("./docs/manual.pdf").resolve()]  # 先测试一个；批量时追加多个 PDF
OUTPUT_DIR = REPO_ROOT / "outputs"
OCR_DIR = OUTPUT_DIR / "ocr_text"
DIFY_DIR = OUTPUT_DIR / "dify_upload"
for directory in (OUTPUT_DIR, OCR_DIR, DIFY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DOCUMENT_TYPE = "manual"  # manual / policy / contract / faq / generic
print(PDF_FILES)


## 1. PaddleOCR 本地 OCR 函数

下面直接在 notebook 中调用 pip 安装后的 PaddleOCR，不再走 HTTP 接口。

处理方式：
- 使用 PyMuPDF 把 PDF 每页渲染为 PNG。
- 使用 PaddleOCR 识别每页文字。
- 输出 `[(page_number, text)]`，后续直接交给分块逻辑。

如果 PDF 是文本型，也可以跳过 OCR，直接使用解析器读取 PDF 文本。


In [ ]:
import fitz  # PyMuPDF
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(use_angle_cls=True, lang="ch")

def ocr_pdf_with_paddle(pdf_path: Path, image_dpi: int = 180):
    pages = []
    zoom = image_dpi / 72
    matrix = fitz.Matrix(zoom, zoom)
    with fitz.open(pdf_path) as document:
        for page_index, page in enumerate(document, start=1):
            pixmap = page.get_pixmap(matrix=matrix, alpha=False)
            image_path = OCR_DIR / f"{pdf_path.stem}_page_{page_index}.png"
            pixmap.save(image_path)
            result = ocr_engine.ocr(str(image_path), cls=True)
            lines = []
            for block in result or []:
                for line in block or []:
                    if len(line) >= 2 and line[1]:
                        lines.append(line[1][0])
            pages.append((page_index, "\n".join(lines)))
    return pages


## 2. 文本抽取 + OCR fallback + 分块

逻辑：
1. 先用 `RuleDocumentParser` 尝试读取 PDF 文本。
2. 如果文本型 PDF 能抽出内容，直接使用。
3. 如果抽取为空，则调用上面的 PaddleOCR 函数，再复用解析器的 `_split_blocks`、`_section_heading`、`_chunk_text` 组织 chunks。

这样 notebook 中能清楚看到 OCR 与分块边界。


In [ ]:
from data_agent import DocumentChunk, DocumentMetadata, RuleDocumentParser

def parse_pdf_for_review(pdf_path: Path, document_type: str = DOCUMENT_TYPE):
    parser = RuleDocumentParser(document_type=document_type)
    chunks = parser.parse(
        pdf_path,
        document_format="pdf",
        extra_metadata={"document_type": document_type, "source_kind": "notebook_review"},
    )
    if chunks:
        return chunks, "pdf_text"

    pages = ocr_pdf_with_paddle(pdf_path)
    metadata = DocumentMetadata.from_path(
        pdf_path,
        {"document_type": document_type, "document_format": "pdf", "extraction_method": "paddleocr", "source_kind": "notebook_review"},
    )
    ocr_chunks = []
    current_section = None
    for page_number, text in pages:
        for block in parser._split_blocks(text):
            heading = parser._section_heading(block)
            if heading:
                current_section = heading
            for piece in parser._chunk_text(block):
                ocr_chunks.append(DocumentChunk(piece, metadata, len(ocr_chunks), current_section, page_number))
    return ocr_chunks, "paddleocr"

all_chunks_by_pdf = {}
for pdf_path in PDF_FILES:
    chunks, method = parse_pdf_for_review(pdf_path)
    all_chunks_by_pdf[pdf_path] = chunks
    print(pdf_path.name, method, len(chunks))
    for chunk in chunks[:2]:
        print(chunk.metadata_payload())
        print(chunk.text[:300])
        print("---")


## 3. 目录解析检查

针对目录页形如 `1.1 人员防护 ........ 1-1` 的内容，可以抽取目录结构，用于 review 或作为后续 metadata。


In [ ]:
from data_agent import parse_toc_entries

for pdf_path, chunks in all_chunks_by_pdf.items():
    toc_text = "\n".join(chunk.text for chunk in chunks[:10])
    toc_entries = parse_toc_entries(toc_text)
    print(pdf_path.name, "toc entries:", len(toc_entries))
    for entry in toc_entries[:10]:
        print(entry)


## 4. 导出本地 Dify 上传文件

当前版本以“导出本地后上传 Dify”为主，不直接调用 Dify API。

每个 PDF 输出两类文件：
- `.txt`：适合直接上传 Dify Knowledge。
- `.jsonl`：保留 chunk metadata，便于检查或后续脚本导入。


In [ ]:
from data_agent import write_dify_jsonl, write_dify_text

for pdf_path, chunks in all_chunks_by_pdf.items():
    txt_path = DIFY_DIR / f"{pdf_path.stem}.dify.txt"
    jsonl_path = DIFY_DIR / f"{pdf_path.stem}.dify.jsonl"
    txt_count = write_dify_text(chunks, txt_path)
    jsonl_count = write_dify_jsonl(chunks, jsonl_path)
    print(pdf_path.name, "txt chunks:", txt_count, txt_path)
    print(pdf_path.name, "jsonl chunks:", jsonl_count, jsonl_path)


## 5. 可选：写入 Milvus 功能块

这一块保留为可选测试：使用本地 BGE embedding，将同一批 chunks 写入 Milvus。

如果只是验证 Dify 上传文件，可以不执行入库代码。


In [ ]:
from data_agent import LocalBGEEmbeddingProvider, MilvusRuleKnowledgeBase

BGE_API_URL = "http://localhost:8000/v1/embeddings"
MILVUS_URI = "http://localhost:19530"
COLLECTION_NAME = "rule_knowledge_base"

embedding_provider = LocalBGEEmbeddingProvider(api_url=BGE_API_URL, model="bge-m3", dimension=1024)
kb = MilvusRuleKnowledgeBase(MILVUS_URI, None, COLLECTION_NAME, embedding_provider)

# 取消注释后真正写入 Milvus。
# for pdf_path, chunks in all_chunks_by_pdf.items():
#     upserted = kb.upsert_chunks(chunks)
#     print(pdf_path.name, "upserted:", upserted)


## 6. 批量 PDF 使用方式

把第 0 步的 `PDF_FILES` 改成多个 PDF 即可：

```python
PDF_FILES = sorted(Path("./docs").glob("*.pdf"))
```

后续 OCR、分块、Dify 文件导出和 Milvus 入库都会按 PDF 循环处理。
